# Exploratory Data Analysis (EDA)

### Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [7]:
# Adding dataset to the project
DATA_PATH = 'Data/september_2026.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)


## Understanding the dataset

In [70]:
# Standardise column names
df.columns = df.columns.astype(str).str.strip().str.lower()

print("NMG HOUSEHOLD SURVEY DATASET OVERVIEW")
print("-" * 40)

print(f"Total respondents/rows: {df.shape[0]:,}")
print(f"Total variables: {df.shape[1]:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

print("\nDATA TYPES")
print(df.dtypes.astype(str).value_counts())

print("\nFIRST FIVE RECORDS")
display(df.head())

print("\nOVERALL MISSING DATA")
print(f"Missing cells: {df.isna().sum().sum():,}")
print(f"Missing percentage: {df.isna().mean().mean():.2%}")

NMG HOUSEHOLD SURVEY DATASET OVERVIEW
----------------------------------------
Total respondents/rows: 6,118
Total variables: 554
Duplicate rows: 0

DATA TYPES
float64    458
int64       96
Name: count, dtype: int64

FIRST FIVE RECORDS


,we_factor,wave,fgenage_1,dage,dregion,fgen,qhousing,dhousing,qual,be2afree_1,...,col06_rank_2,col06_rank_3,col06_rank_4,col06_rank_5,col06_rank_6,col06_rank_7,col06_rank_8,col06_rank_9,col06_rank_10,col06_rank_970
0,0.953897,31,67,6,9,1,2,2,4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.912132,31,63,5,6,1,2,2,3,12000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.091621,31,75,6,5,1,2,2,4,250.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.829547,31,80,6,4,2,2,2,4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.243876,31,62,5,7,1,5,3,3,40.0,...,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



OVERALL MISSING DATA
Missing cells: 2,103,662
Missing percentage: 62.07%


## Pick and explore quality of shortlisted variables
I first typed keywords like "age" and "income' into the excel description document explaining the codes for each survey question.

In [73]:
# Shortlisted variables based on earlier analysis
#qbe15

variables = {
    "uq03": ("Unsecured loan/credit arrears in past 12 months", "Primary target"),
    "boe60": ("Mortgage arrears", "Alternative target"),
    "fq01": ("Change in household financial situation", "EDA / alternative target"),
    "fgenage_1": ("Respondent age group", "Predictor"),
    "dage": ("Respondent age", "Predictor"),
    "fgen": ("Respondent gender", "Predictor"),
    "dregion": ("UK region", "Predictor"),
    "qhousing": ("Housing tenure", "Predictor"),
    "nochildren": ("Number of children under 16", "Predictor"),
    "recent_workingstatus": ("Previous employment status", "Predictor"),
    "overallincomecomb": ("Combined household income bands", "Predictor"),
    "bc10a": ("Change in annual household income", "Predictor"),
    "boe65a": ("Household cash savings bands", "Predictor"),
    "be16rea": ("Outstanding household loan balances", "Predictor"),
    "sv11": ("Change in saving behaviour", "Predictor"),
    "qbe22b": ("Concern about current debt level", "Predictor"),
    "col03": ("Cost-of-living spending adjustments", "Predictor"),
    "fq02": ("Expected future financial position", "EDA / optional predictor"),
    "we_factor_adjusted": ("Survey adjustment weight", "Survey weighting"),
    #"qmarital": "Marital Status", false
    "socialgrade": ("Social Grade", "Predictor"),
    #"qincome": "Income", false
    #"qincomev1": "Household Income", false
    "qual": ("Highest Qualification", "Predictor"),
    "nat1a": ("Nationality", "Predictor"),
    "nat1b1": ("Year moved to UK", "Predictor"),
    "qregion" : ("Region you live in", "Predictor"),
    #"bc2": "Government benefits received" false
    "overallbe08": ("monthly mortgage payments", "Predictor"),
    "js2_1": ("Within the last week (7 days), how many hours did you spend searching and applying for jobs? This can include looking for job-postings, sending applications, or designing a CV.", "Predictor")
}

# Add individual fq01b columns automatically
fq01b_descriptions = {
    "fq01b_1": "Fallen behind on mortgage payments",
    "fq01b_3": "Fallen behind on rent but allowed to remain",
    "fq01b_4": "Fallen behind on rent and/or asked to leave",
    "fq01b_5": "Fallen behind on unsecured loan/credit payments",
    "fq01b_6": "Fallen behind on bills or utilities",
    "fq01b_7": "Increased overdraft",
    "fq01b_8": "Increased credit card balance",
    "fq01b_9": "Applied for an unsecured loan",
    "fq01b_10": "Other financial difficulty",
    "fq01b_11": "No reported financial difficulty",
    "fq01b_12": "Don't know",
    "fq01b_13": "Prefer not to state"
}

In [86]:
import re

# Read the survey description sheet
description = pd.read_excel(
    "Data/NMG_Data_Description.xlsx",
    header=None
).iloc[:, 0].fillna("").astype(str)

# Extract response codes and labels
question_mappings = {}
current_variable = None

for row in description:
    row = row.strip()

    # Find question heading, e.g. uq03 - ...
    match = re.match(r"^([A-Za-z][A-Za-z0-9_]*)\s*[-–]\s", row)

    if match:
        current_variable = match.group(1).lower()
        if current_variable in variables:
            question_mappings[current_variable] = {}
        continue

    # Find coded answer, e.g. m Yes (1)
    match = re.match(r"^[mq]\s+(.+?)\s*\((\d+)\)", row)

    if match and current_variable in variables:
        label, code = match.groups()
        question_mappings[current_variable][int(code)] = label.strip()

# Numeric values should remain numeric
numeric_variables = [
    "fgenage_1",
    "nochildren",
    "js2_1",
    "we_factor_adjusted"
]

for variable in numeric_variables:
    question_mappings.pop(variable, None)

# Display the mappings
for variable, mapping in question_mappings.items():
    print(f"\n{variable}: {variables[variable][0]}")
    print(mapping)


dage: Respondent age
{1: '18 - 24', 2: '25 - 34', 3: '35 - 44', 4: '45 - 54', 5: '55 - 64', 6: '65+'}

qregion: Region you live in
{1: 'North', 2: 'North West', 3: 'Yorks & Humberside', 4: 'West Midlands', 5: 'East Midlands', 6: 'East Anglia', 7: 'South West', 8: 'South East', 9: 'Greater London', 10: 'Wales', 11: 'Scotland', 12: 'Northern Ireland', 99: 'Not On Map', 13: 'Non UK'}

dregion: UK region
{1: 'North / Yorks & Humberside', 2: 'North West', 3: 'Midlands', 4: 'South East', 5: 'East Anglia', 6: 'South West', 7: 'Wales', 8: 'London', 9: 'Scotland'}

fgen: Respondent gender
{1: 'Male', 2: 'Female'}

qhousing: Housing tenure
{1: 'Being bought on a mortgage', 2: 'Owned outright by household', 3: 'Rented from local authority', 4: 'Rented from a private landlord', 5: 'Belongs to housing association', 98: 'Other (please specify)', 96: 'Prefer not to state'}

recent_workingstatus: Previous employment status
{1: 'Have / had paid job - working full time (30+ hours per week)', 2: 'Have /

In [87]:
# Example: map income survey codes to their descriptions
df["income_label"] = (
    pd.to_numeric(df["overallincomecomb"], errors="coerce")
    .map(question_mappings["overallincomecomb"])
)

# Example: map housing tenure
df["housing_label"] = (
    pd.to_numeric(df["qhousing"], errors="coerce")
    .map(question_mappings["qhousing"])
)

/var/folders/cc/xzk8m4152gg0hn0wv4f7djc80000gn/T/ipykernel_36575/2721039261.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["income_label"] = (
/var/folders/cc/xzk8m4152gg0hn0wv4f7djc80000gn/T/ipykernel_36575/2721039261.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["housing_label"] = (


In [93]:
for variable, (description, role) in variables.items():
    print(f"\n{'+' * 60}")
    print(f"Variable: {variable}")
    print(f"Description: {description}")
    print(f"Role: {role}")

    mapping = question_mappings.get(variable, {})

    if mapping:
        for code, answer in mapping.items():
            print(f"  {code}: {answer}")
    else:
        print("  No mapping found (or numeric variable)")


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Variable: uq03
Description: Unsecured loan/credit arrears in past 12 months
Role: Primary target
  1: Yes
  2: No
  3: Don't know
  4: Prefer not to state

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Variable: boe60
Description: Mortgage arrears
Role: Alternative target
  1: Yes
  2: No
  3: Don't know
  4: Prefer not to state

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Variable: fq01
Description: Change in household financial situation
Role: EDA / alternative target
  1: Got a lot better
  2: Got a little better
  3: Stayed the same
  4: Got a little worse
  5: Got a lot worse
  6: Don't Know
  7: Prefer not to state

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Variable: fgenage_1
Description: Respondent age group
Role: Predictor
  No mapping found (or numeric variable)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Variable: dage
Description: Respondent age

In [ ]:
# Populated does include answers like "Don't know" or "Prefer not to say" and will need filtering out at a later stage.
selected = df[list(variables)]

coverage = pd.DataFrame({
    "Populated": selected.notna().sum(),
    "Missing %": selected.isna().mean().mul(100).round(1),
    "Distinct stored values": selected.nunique()
})

coverage.sort_values("Missing %", ascending=False)

,Populated,Missing %,Distinct stored values
be16rea,421,93.1,28
nat1b1,445,92.7,71
overallbe08,1606,73.7,26
boe60,1606,73.7,4
col03,4034,34.1,6
qbe22b,4235,30.8,5
nat1a,6064,0.9,82
uq03,6118,0.0,4
qregion,6118,0.0,11
qual,6118,0.0,8


In [79]:
print(df["dage"].dtype)
print(df["dage"].unique())

int64
[6 5 4 3 2 1]


In [81]:
# Comparing age band var with actual age var to see if they match up. The age band var is more useful for analysis, but the actual age var is more useful for modelling.
age_band_labels = {
    1: "18-24",
    2: "25-34",
    3: "35-44",
    4: "45-54",
    5: "55-64",
    6: "65+"
}

age_summary = df.groupby("dage")["fgenage_1"].agg(
    respondents="size",
    youngest="min",
    oldest="max",
    median_age="median"
)

age_summary.insert(0, "age_band", age_summary.index.map(age_band_labels))
age_summary

,age_band,respondents,youngest,oldest,median_age
dage,,,,,
1,18-24,622,18,24,21.0
2,25-34,945,25,34,30.0
3,35-44,851,35,44,40.0
4,45-54,1003,45,54,50.0
5,55-64,1079,55,64,60.0
6,65+,1618,65,97,72.0


# TODO
- <p style="color:yellow"> Add fully mapped answers to dict<p>
- Add EDA distribution
- Add graphs for vars so far
- Trace and explain missingness
- Clean data
- Prep for final clean dataset
- Export clean consolidated dataset

## After that:
- Engineer variables
- Train initial models
- Eval
- Retrain